In [ ]:
"""
Merge Banja Luka, Bijeljina, Brod, Doboj, Gacko, Prijedor, Trebinje, Ugljevik
air-quality files into the common FHZ-style schema:

station, pm10_1h, pm25_1h, so2_1h, no2_1h, o3_1h, co_1h, city, year,
datetime, month, day, hour, wind_speed, temperature, season
"""

import re
import datetime as dt
import pandas as pd
import openpyxl
import xlrd

# --- Colab setup ---------------------------------------------------------
# If running in Google Colab, mount Drive and point DATA_DIR at the shared
# folder. If the folder is under "Shared with me" rather than "My Drive",
# add a shortcut to it first (right-click the folder in Drive ->
# "Organize" -> "Add shortcut to Drive"), otherwise it won't be visible
# under MyDrive after mounting.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass  # not running in Colab

TARGET_COLS = [
    "station", "pm10_1h", "pm25_1h", "so2_1h", "no2_1h", "o3_1h", "co_1h",
    "city", "year", "datetime", "month", "day", "hour",
    "wind_speed", "temperature", "season",
]

MISSING_MARKERS = {"-", "--", "", " ", "n/a", "N/A", "na"}


def season_from_month(m):
    if m in (12, 1, 2):
        return "winter"
    if m in (3, 4, 5):
        return "spring"
    if m in (6, 7, 8):
        return "summer"
    return "autumn"


def clean_num(v):
    """Convert a raw cell value to float, handling text markers and stray spaces."""
    if v is None:
        return None
    if isinstance(v, (int, float)):
        return float(v)
    if isinstance(v, str):
        s = v.strip()
        if s in MISSING_MARKERS:
            return None
        s = s.replace(" ", "")  # fixes '49. 57' -> '49.57'
        try:
            return float(s)
        except ValueError:
            return None
    return None


def parse_hourly_datetime(v):
    """Parse the 'D.M.YYYY H:MM' text format used in the hourly Time column."""
    if not isinstance(v, str):
        return None
    s = v.strip()
    m = re.match(r"^(\d{1,2})\.(\d{1,2})\.(\d{4})\s+(\d{1,2}):(\d{2})$", s)
    if not m:
        return None
    day, month, year, hour, minute = map(int, m.groups())
    try:
        parsed = dt.datetime(year, month, day, hour % 24, minute)
    except ValueError:
        return None
    if parsed.year < 2015:  # guards against stray source typos (e.g. Prijedor '1.3.2012')
        return None
    return parsed


def excel_serial_to_date(v):
    """Bijeljina stores Datum as an Excel serial number (days since 1899-12-30)."""
    if v is None:
        return None
    if isinstance(v, (int, float)):
        return dt.datetime(1899, 12, 30) + dt.timedelta(days=v)
    return None


# ---------------------------------------------------------------------------
# Per-city configuration: source column index (0 = Time/Datum column) -> field
# Row indices are 1-based for openpyxl, 0-based for xlrd, set per engine below.
# ---------------------------------------------------------------------------

CITY_CONFIGS = {
    "Brod": {
        "file": "Brod.xlsx", "engine": "openpyxl", "hourly": True,
        "header_rows": 3,  # data starts at row 4 (1-based)
        "col_map": {1: "pm10_1h", 2: "temperature", 7: "wind_speed",
                    8: "co_1h", 9: "no2_1h", 10: "o3_1h", 11: "so2_1h", 13: "pm25_1h"},
    },
    "Banja Luka": {
        "file": "Banja Luka.xlsx", "engine": "openpyxl", "hourly": True,
        "header_rows": 3,
        "col_map": {1: "pm25_1h", 2: "pm10_1h", 3: "temperature", 5: "wind_speed",
                    8: "co_1h", 10: "no2_1h", 12: "o3_1h", 13: "so2_1h"},
    },
    "Doboj": {
        "file": "Doboj.xlsx", "engine": "openpyxl", "hourly": True,
        "header_rows": 3,
        "col_map": {2: "no2_1h", 4: "so2_1h", 8: "wind_speed",
                    10: "pm10_1h", 11: "o3_1h", 12: "co_1h"},
    },
    "Gacko": {
        "file": "Gacko.xls", "engine": "xlrd", "hourly": True,
        "header_rows": 3,
        "col_map": {1: "pm10_1h", 3: "no2_1h", 5: "so2_1h", 8: "temperature", 9: "wind_speed"},
    },
    "Prijedor": {
        "file": "Prijedor.xlsx", "engine": "openpyxl", "hourly": True,
        "header_rows": 3,
        "col_map": {1: "so2_1h", 2: "no2_1h", 5: "co_1h", 6: "o3_1h",
                    7: "pm10_1h", 8: "pm25_1h", 9: "wind_speed", 11: "temperature"},
    },
    "Trebinje": {
        "file": "Trebinje.xlsx", "engine": "openpyxl", "hourly": True,
        "header_rows": 3,
        "col_map": {2: "pm25_1h", 4: "pm10_1h", 5: "temperature", 7: "wind_speed",
                    10: "co_1h", 12: "no2_1h", 14: "o3_1h", 15: "so2_1h"},
    },
    "Ugljevik": {
        "file": "Ugljevika.xlsx", "engine": "openpyxl", "hourly": True,
        "header_rows": 3,
        "col_map": {1: "temperature", 6: "wind_speed", 8: "no2_1h",
                    10: "so2_1h", 11: "pm10_1h"},
    },
    "Bijeljina": {
        "file": "Bijeljina.xls", "engine": "xlrd", "hourly": False,
        "header_rows": 3,
        "col_map": {1: "so2_1h", 2: "co_1h", 3: "no2_1h", 6: "o3_1h",
                    7: "pm25_1h", 8: "pm10_1h", 11: "wind_speed", 13: "temperature"},
    },
}

DATA_DIR = "/content/drive/MyDrive/air_pollution_bih/dataset/rhz_data/"


def load_city_openpyxl(city, cfg):
    wb = openpyxl.load_workbook(DATA_DIR + cfg["file"], data_only=True)
    rows = []
    for sheet_name in wb.sheetnames:
        ws = wb[sheet_name]
        start_row = cfg["header_rows"] + 1
        for r in range(start_row, ws.max_row + 1):
            time_val = ws.cell(row=r, column=1).value
            parsed_dt = parse_hourly_datetime(time_val)
            if parsed_dt is None:
                continue  # skip stray/trailing rows that aren't real data
            row = {"datetime": parsed_dt}
            for col_idx, field in cfg["col_map"].items():
                raw = ws.cell(row=r, column=col_idx + 1).value  # +1: col 0 = Time
                row[field] = clean_num(raw)
            rows.append(row)
    return rows


def load_city_xlrd_hourly(city, cfg):
    wb = xlrd.open_workbook(DATA_DIR + cfg["file"])
    rows = []
    for sheet_name in wb.sheet_names():
        ws = wb.sheet_by_name(sheet_name)
        start_row = cfg["header_rows"]  # 0-based
        for r in range(start_row, ws.nrows):
            time_val = ws.cell_value(r, 0)
            parsed_dt = parse_hourly_datetime(time_val)
            if parsed_dt is None:
                continue
            row = {"datetime": parsed_dt}
            for col_idx, field in cfg["col_map"].items():
                raw = ws.cell_value(r, col_idx)
                row[field] = clean_num(raw)
            rows.append(row)
    return rows


def load_city_xlrd_daily(city, cfg):
    """Bijeljina: daily data, Datum column is an Excel serial number."""
    wb = xlrd.open_workbook(DATA_DIR + cfg["file"])
    rows = []
    for sheet_name in wb.sheet_names():
        ws = wb.sheet_by_name(sheet_name)
        start_row = cfg["header_rows"]  # 0-based
        for r in range(start_row, ws.nrows):
            date_val = ws.cell_value(r, 0)
            parsed_dt = excel_serial_to_date(date_val)
            if parsed_dt is None:
                continue
            row = {"datetime": parsed_dt}
            for col_idx, field in cfg["col_map"].items():
                raw = ws.cell_value(r, col_idx)
                row[field] = clean_num(raw)
            rows.append(row)
    return rows


def build_city_df(city, cfg):
    if cfg["engine"] == "openpyxl":
        rows = load_city_openpyxl(city, cfg)
    elif cfg["engine"] == "xlrd" and cfg["hourly"]:
        rows = load_city_xlrd_hourly(city, cfg)
    else:
        rows = load_city_xlrd_daily(city, cfg)

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    for col in ["pm10_1h", "pm25_1h", "so2_1h", "no2_1h", "o3_1h", "co_1h",
                "wind_speed", "temperature"]:
        if col not in df.columns:
            df[col] = pd.NA

    df["station"] = city
    df["city"] = city
    df["year"] = df["datetime"].dt.year
    df["month"] = df["datetime"].dt.month
    df["day"] = df["datetime"].dt.day
    df["hour"] = df["datetime"].dt.hour if cfg["hourly"] else pd.NA
    df["season"] = df["month"].apply(season_from_month)

    return df[TARGET_COLS]


def main():
    all_dfs = []
    for city, cfg in CITY_CONFIGS.items():
        print(f"Processing {city} ...")
        df = build_city_df(city, cfg)
        print(f"  -> {len(df)} rows")
        all_dfs.append(df)

    merged = pd.concat(all_dfs, ignore_index=True)
    merged = merged.sort_values(["city", "datetime"]).reset_index(drop=True)
    print("\nTotal merged rows:", len(merged))
    print(merged.groupby("city").size())
    merged.to_csv(DATA_DIR + "rhz_data.csv", index=False)
    merged.to_excel(DATA_DIR + "rhz_data.xlsx", index=False)
    print("\nSaved merged_air_quality.csv / .xlsx")
    print(merged.head(10))
    print(merged.isna().sum())


if __name__ == "__main__":
    main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Processing Brod ...
  -> 43824 rows
Processing Banja Luka ...
  -> 43824 rows
Processing Doboj ...
  -> 35064 rows
Processing Gacko ...
  -> 43824 rows
Processing Prijedor ...
  -> 43823 rows
Processing Trebinje ...
  -> 43824 rows
Processing Ugljevik ...
  -> 43824 rows
Processing Bijeljina ...
  -> 1708 rows


/tmp/ipykernel_3436/1274727327.py:243: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  merged = pd.concat(all_dfs, ignore_index=True)



Total merged rows: 299715
city
Banja Luka    43824
Bijeljina      1708
Brod          43824
Doboj         35064
Gacko         43824
Prijedor      43823
Trebinje      43824
Ugljevik      43824
dtype: int64

Saved merged_air_quality.csv / .xlsx
      station  pm10_1h  pm25_1h  so2_1h  no2_1h  o3_1h  co_1h        city  \
0  Banja Luka      NaN      NaN     NaN     NaN    NaN    NaN  Banja Luka   
1  Banja Luka      NaN      NaN     NaN     NaN    NaN    NaN  Banja Luka   
2  Banja Luka      NaN      NaN     NaN     NaN    NaN    NaN  Banja Luka   
3  Banja Luka      NaN      NaN     NaN     NaN    NaN    NaN  Banja Luka   
4  Banja Luka      NaN      NaN     NaN     NaN    NaN    NaN  Banja Luka   
5  Banja Luka      NaN      NaN     NaN     NaN    NaN    NaN  Banja Luka   
6  Banja Luka      NaN      NaN     NaN     NaN    NaN    NaN  Banja Luka   
7  Banja Luka      NaN      NaN     NaN     NaN    NaN    NaN  Banja Luka   
8  Banja Luka      NaN      NaN     NaN     NaN    NaN    NaN  B